In [37]:
from eda.eda_utils import load_data, replace_with_nan, minmax_stretch, percentile_stretch, plot_band, calculate_band_statistics, standardize, histogram_before_after_standardization_z_score,show_band_vs_z, correlation_matrix, correlation_plot

import copy
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import cmocean as cmo
from pandas import DataFrame

import spectral as spy
import re


In [ ]:
"""
Problem 1:
    Installed spectral python and used spectral.envi.open to read through the tait_labsphere.hdr file.
"""

In [18]:
sentinel2_data = load_data('/Users/wangtiles/Applcns_of_ML_remote_sensing/ml/data_downloaded/sentinel2_rochester.npy')
print("sentinel2_rochester.npy (H, W, B)/(rows, samples, bands) : ",sentinel2_data.shape)
hdr = spy.envi.open('/Users/wangtiles/Applcns_of_ML_remote_sensing/ml/data_downloaded/tait_labsphere.hdr')
print(hdr)
arr = hdr.load()
print(arr.shape) # (H, W, B) 272 bands, definitely cannot just build a correlation matrix


sentinel2_rochester.npy (H, W, B)/(rows, samples, bands) :  (954, 716, 12)
	Data Source:   '/Users/wangtiles/Applcns_of_ML_remote_sensing/ml/data_downloaded/tait_labsphere'
	# Rows:           1039
	# Samples:        1087
	# Bands:           272
	Interleave:        BSQ
	Quantization:  32 bits
	Data format:   float32
(1039, 1087, 272)


In [25]:
print (type(hdr))
print(hdr.metadata.keys())
print(hdr.metadata.values())

<class 'spectral.io.bsqfile.BsqFile'>
dict_keys(['description', 'samples', 'lines', 'bands', 'header offset', 'file type', 'data type', 'interleave', 'sensor type', 'byte order', 'map info', 'coordinate system string', 'wavelength units', 'band names'])
dict_values(['File Resize Result, x resize factor: 1.000000, y resize factor: 1.000000.\n[Mon Sep 15 09:41:58 2025]', '1087', '1039', '272', '0', 'ENVI Standard', '4', 'bsq', 'Unknown', '0', ['Geographic Lat/Lon', '1.0000', '1.0000', '-77.50580812', '43.14056592', '5.1625528480e-07', '3.7805331830e-07', 'WGS-84', 'units=Degrees'], ['GEOGCS["GCS_WGS_1984"', 'DATUM["D_WGS_1984"', 'SPHEROID["WGS_1984"', '6378137.0', '298.257223563]]', 'PRIMEM["Greenwich"', '0.0]', 'UNIT["Degree"', '0.0174532925199433]]'], 'Unknown', ['Resize (398.573 nm:tait_hsi)', 'Resize (400.799 nm:tait_hsi)', 'Resize (403.025 nm:tait_hsi)', 'Resize (405.251 nm:tait_hsi)', 'Resize (407.477 nm:tait_hsi)', 'Resize (409.703 nm:tait_hsi)', 'Resize (411.929 nm:tait_hsi)', 'R

In [39]:
# wavelength units has Resize things, those are the wavelengths for each spectral band.
wavelengths = hdr.metadata.get('wavelength units')
wavelengths

'Unknown'

In [36]:
for key, value in hdr.metadata.items():
    print(key, ":", value)

description : File Resize Result, x resize factor: 1.000000, y resize factor: 1.000000.
[Mon Sep 15 09:41:58 2025]
samples : 1087
lines : 1039
bands : 272
header offset : 0
file type : ENVI Standard
data type : 4
interleave : bsq
sensor type : Unknown
byte order : 0
map info : ['Geographic Lat/Lon', '1.0000', '1.0000', '-77.50580812', '43.14056592', '5.1625528480e-07', '3.7805331830e-07', 'WGS-84', 'units=Degrees']
coordinate system string : ['GEOGCS["GCS_WGS_1984"', 'DATUM["D_WGS_1984"', 'SPHEROID["WGS_1984"', '6378137.0', '298.257223563]]', 'PRIMEM["Greenwich"', '0.0]', 'UNIT["Degree"', '0.0174532925199433]]']
wavelength units : Unknown
band names : ['Resize (398.573 nm:tait_hsi)', 'Resize (400.799 nm:tait_hsi)', 'Resize (403.025 nm:tait_hsi)', 'Resize (405.251 nm:tait_hsi)', 'Resize (407.477 nm:tait_hsi)', 'Resize (409.703 nm:tait_hsi)', 'Resize (411.929 nm:tait_hsi)', 'Resize (414.155 nm:tait_hsi)', 'Resize (416.381 nm:tait_hsi)', 'Resize (418.607 nm:tait_hsi)', 'Resize (420.833 nm

In [48]:
band_names = hdr.metadata.get('band names')
wavelengths = [] # storing all the extracted wavelengths for each of the spectral band

for bn in band_names:
    match = re.search(r'([\d.]+)\s*nm', bn) # 458.674 nm
    if match:
        wavelengths.append(float(match.group(1)))
print("number of bands: ", len(wavelengths), "\nshape(rows*cols / H*W) of each band: ", hdr.shape[:2])
print(wavelengths)

number of bands:  272 
shape(rows*cols / H*W) of each band:  (1039, 1087)
[398.573, 400.799, 403.025, 405.251, 407.477, 409.703, 411.929, 414.155, 416.381, 418.607, 420.833, 423.059, 425.285, 427.511, 429.737, 431.963, 434.188, 436.414, 438.64, 440.866, 443.092, 445.318, 447.544, 449.77, 451.996, 454.222, 456.448, 458.674, 460.9, 463.126, 465.352, 467.578, 469.804, 472.03, 474.256, 476.482, 478.708, 480.934, 483.16, 485.386, 487.612, 489.838, 492.064, 494.29, 496.516, 498.742, 500.968, 503.194, 505.42, 507.646, 509.872, 512.098, 514.324, 516.55, 518.776, 521.002, 523.228, 525.454, 527.68, 529.906, 532.132, 534.358, 536.584, 538.81, 541.036, 543.261, 545.487, 547.713, 549.939, 552.165, 554.391, 556.617, 558.843, 561.069, 563.295, 565.521, 567.747, 569.973, 572.199, 574.425, 576.651, 578.877, 581.103, 583.329, 585.555, 587.781, 590.007, 592.233, 594.459, 596.685, 598.911, 601.137, 603.363, 605.589, 607.815, 610.041, 612.267, 614.493, 616.719, 618.945, 621.171, 623.397, 625.623, 627.849, 